<h1 style="color:DodgerBlue">Индивидуальный проект</h1>

## Отзывы о продуктах и услугах
**Вариант №18. Студент: Чирченко Максим.**

Продолжение Sprint2: `Review → ProductReview`, `Review → ServiceReview → ServiceTimeReview`.
Сохранены интерфейсы `IRateable`, `IPublishable` и операции предыдущего спринта. Ноутбук содержит всю реализацию и запускается самостоятельно.

### Дополнительное задание
Добавить к существующим базовому и производным классам по 3–4 атрибута и метода; реализовать полиморфизм с перекрытием и перегрузкой методов, а также generic-классы.

| Класс | Новые свойства | Новые методы |
|---|---|---|
| Review | Title, Language, UpdatedAt | RenameTitle, SetLanguage, EditText(string, string), GetReviewDetails(bool) |
| ProductReview | Pros, Cons, WouldRecommend | SetPros, SetCons, Recommend, RateProduct(int, int) |
| ServiceReview | Location, BookingCode, WouldReturn | SetLocation, SetBookingCode, RecommendService |
| ServiceTimeReview | ScheduledStartTime, ActualStartTime, HasDelay | SetArrival, CalculateDelay, ResetArrival |

`DisplayReview()` и `GetReviewDetails()` переопределены в наследниках. Вызовы через `Review` и `ReviewCollection<Review>` выбирают реализации реального типа.
`ReviewCollection<ProductReview>` принимает только отзывы о товарах. Перегрузка `RateProduct(int, int)` вычисляет среднюю оценку качества и цены с округлением половин вверх.
Время относится к одному дню; ожидание клиента и задержка фактического начала учитываются отдельно.

### Реализация

In [1]:
using System;
using System.Collections.Generic;

public interface IRateable
{
    int Rating { get; }
    void RateProduct(int rating);
}
public interface IPublishable
{
    bool IsPublished { get; }
    void Publish();
    void Hide();
}

// Общая основа отзывов и реализация двух независимых контрактов.
public abstract class Review : IRateable, IPublishable
{
    public string Title { get; private set; } = "Отзыв";
    public string Language { get; private set; } = "ru";
    public DateTime UpdatedAt { get; private set; }
    public void RenameTitle(string title)
    {
        if (string.IsNullOrWhiteSpace(title)) throw new ArgumentException("Заголовок обязателен.", nameof(title));
        Title = title;
        UpdatedAt = DateTime.Now;
    }
    public void SetLanguage(string language)
    {
        if (string.IsNullOrWhiteSpace(language)) throw new ArgumentException("Язык обязателен.", nameof(language));
        Language = language;
        UpdatedAt = DateTime.Now;
    }
    public void EditText(string title, string text)
    {
        if (string.IsNullOrWhiteSpace(title) || string.IsNullOrWhiteSpace(text))
            throw new ArgumentException("Заголовок и текст обязательны.");
        RenameTitle(title);
        EditText(text);
    }
    // Перегрузка вызывает виртуальный метод без параметров.
    public string GetReviewDetails(bool brief) => brief ? $"{Title}: {Rating}/5" : GetReviewDetails();
    public int ReviewId { get; }
    public string Text { get; private set; }
    public int Rating { get; private set; }
    public string Author { get; set; }
    public DateTime CreatedAt { get; }
    public bool IsPublished { get; private set; }
    public int HelpfulVotes { get; private set; }

    protected Review(int reviewId, string text, int rating, string author)
    {
        ReviewId = reviewId;
        EditText(text);
        RateProduct(rating);
        Author = author;
        CreatedAt = DateTime.Today;
    }
    public virtual void RateProduct(int rating)
    {
        if (rating < 1 || rating > 5)
            throw new ArgumentOutOfRangeException(nameof(rating), "Оценка должна быть от 1 до 5.");
        Rating = rating;
    }
    public void EditText(string text)
    {
        if (string.IsNullOrWhiteSpace(text))
            throw new ArgumentException("Текст отзыва не должен быть пустым.", nameof(text));
        Text = text;
        UpdatedAt = DateTime.Now;
    }
    public void Publish() => IsPublished = true;
    public void Hide() => IsPublished = false;
    public void MarkHelpful() => HelpfulVotes++;
    public abstract void DisplayReview();
    public virtual string GetReviewDetails() =>
        $"{Title} [{Language}], обновлён: {UpdatedAt:dd.MM.yyyy HH:mm}; отзыв #{ReviewId}: {Text}; {Rating}/5; автор: {Author}; создан: {CreatedAt:dd.MM.yyyy}; опубликован: {IsPublished}; полезен: {HelpfulVotes}";
}

// Простое наследование: Review -> ProductReview.
public class ProductReview : Review
{
    public string Pros { get; private set; } = "Не указаны";
    public string Cons { get; private set; } = "Не указаны";
    public bool WouldRecommend { get; private set; }
    public void SetPros(string pros)
    {
        if (string.IsNullOrWhiteSpace(pros)) throw new ArgumentException("Укажите достоинства.", nameof(pros));
        Pros = pros;
    }
    public void SetCons(string cons)
    {
        if (string.IsNullOrWhiteSpace(cons)) throw new ArgumentException("Укажите недостатки.", nameof(cons));
        Cons = cons;
    }
    public void Recommend(bool recommend) => WouldRecommend = recommend;
    public void RateProduct(int quality, int valueForMoney)
    {
        if (quality < 1 || quality > 5 || valueForMoney < 1 || valueForMoney > 5)
            throw new ArgumentOutOfRangeException("Оценки должны быть от 1 до 5.");
        base.RateProduct((int)Math.Round((quality + valueForMoney) / 2.0, MidpointRounding.AwayFromZero));
    }
    public string ProductId { get; }
    public string ProductName { get; private set; }
    public DateTime PurchaseDate { get; private set; }
    public bool IsVerifiedPurchase { get; private set; }

    public ProductReview(int id, string text, int rating, string productId,
        string author, string productName, DateTime purchaseDate)
        : base(id, text, rating, author)
    {
        ProductId = productId;
        RenameProduct(productName);
        SetPurchaseDate(purchaseDate);
    }
    public void RenameProduct(string name)
    {
        if (string.IsNullOrWhiteSpace(name)) throw new ArgumentException("Название товара обязательно.", nameof(name));
        ProductName = name;
    }
    public void SetPurchaseDate(DateTime date)
    {
        if (date.Date > DateTime.Today) throw new ArgumentOutOfRangeException(nameof(date), "Дата покупки не может быть в будущем.");
        PurchaseDate = date.Date;
        IsVerifiedPurchase = false;
    }
    public void ConfirmPurchase() => IsVerifiedPurchase = true;
    public override void DisplayReview() => Console.WriteLine($"Товар «{ProductName}»: {Text} ({Rating}/5)");
    public override string GetReviewDetails() =>
        $"{base.GetReviewDetails()}; товар: {ProductId}, {ProductName}; покупка: {PurchaseDate:dd.MM.yyyy}; подтверждена: {IsVerifiedPurchase}; плюсы: {Pros}; минусы: {Cons}; рекомендую: {WouldRecommend}";
}

// Простое наследование: Review -> ServiceReview.
public class ServiceReview : Review
{
    public string Location { get; private set; } = "Не указано";
    public string BookingCode { get; private set; } = "Без записи";
    public bool WouldReturn { get; private set; }
    public void SetLocation(string location)
    {
        if (string.IsNullOrWhiteSpace(location)) throw new ArgumentException("Адрес обязателен.", nameof(location));
        Location = location;
    }
    public void SetBookingCode(string code)
    {
        if (string.IsNullOrWhiteSpace(code)) throw new ArgumentException("Код записи обязателен.", nameof(code));
        BookingCode = code;
    }
    public void RecommendService(bool wouldReturn) => WouldReturn = wouldReturn;
    public DateTime VisitDate { get; private set; }
    public string ServiceName { get; set; }
    public string EmployeeName { get; private set; }
    public decimal ServiceCost { get; private set; }

    public ServiceReview(int id, string text, int rating, DateTime visitDate,
        string author, string serviceName, string employeeName, decimal serviceCost)
        : base(id, text, rating, author)
    {
        VisitDate = visitDate.Date;
        ServiceName = serviceName;
        AssignEmployee(employeeName);
        UpdateServiceCost(serviceCost);
    }
    public void Reschedule(DateTime date) => VisitDate = date.Date;
    public void AssignEmployee(string name)
    {
        if (string.IsNullOrWhiteSpace(name)) throw new ArgumentException("Имя сотрудника обязательно.", nameof(name));
        EmployeeName = name;
    }
    public void UpdateServiceCost(decimal cost)
    {
        if (cost < 0) throw new ArgumentOutOfRangeException(nameof(cost), "Стоимость не может быть отрицательной.");
        ServiceCost = cost;
    }
    public override void DisplayReview() => Console.WriteLine($"Услуга «{ServiceName}» от {VisitDate:dd.MM.yyyy}: {Text} ({Rating}/5)");
    public override string GetReviewDetails() =>
        $"{base.GetReviewDetails()}; услуга: {ServiceName}; посещение: {VisitDate:dd.MM.yyyy}; сотрудник: {EmployeeName}; стоимость: {ServiceCost:F2} руб.; адрес: {Location}; запись: {BookingCode}; вернусь: {WouldReturn}";
}

// Многоуровневое наследование: Review -> ServiceReview -> ServiceTimeReview.
public class ServiceTimeReview : ServiceReview
{
    public TimeSpan ScheduledStartTime { get; private set; }
    public TimeSpan ActualStartTime { get; private set; }
    public bool HasDelay { get; private set; }
    public void SetArrival(TimeSpan scheduled, TimeSpan actual)
    {
        if (scheduled < TimeSpan.Zero || scheduled >= TimeSpan.FromDays(1) ||
            actual < TimeSpan.Zero || actual >= TimeSpan.FromDays(1))
            throw new ArgumentException("Время должно быть в пределах одного дня.");
        ScheduledStartTime = scheduled;
        ActualStartTime = actual;
        HasDelay = actual > scheduled;
    }
    public TimeSpan CalculateDelay() => HasDelay ? ActualStartTime - ScheduledStartTime : TimeSpan.Zero;
    public void ResetArrival() => SetArrival(StartTime, StartTime);
    public TimeSpan StartTime { get; private set; }
    public TimeSpan EndTime { get; private set; }
    public int WaitingMinutes { get; private set; }
    public bool WasOnTime { get; private set; }

    public ServiceTimeReview(int id, string text, int rating, DateTime date, TimeSpan startTime,
        string author, string serviceName, string employeeName, decimal cost, TimeSpan endTime)
        : base(id, text, rating, date, author, serviceName, employeeName, cost)
    {
        RescheduleTime(startTime, endTime);
    }
    // Время начала и окончания относится к одному дню.
    public void RescheduleTime(TimeSpan start, TimeSpan end)
    {
        if (start < TimeSpan.Zero || end >= TimeSpan.FromDays(1) || end <= start)
            throw new ArgumentException("Нужно указать время одного дня, окончание позже начала.");
        StartTime = start;
        EndTime = end;
        WaitingMinutes = 0;
        WasOnTime = true;
        ResetArrival();
    }
    public TimeSpan CalculateDuration() => EndTime - StartTime;
    public void RecordWaiting(int minutes)
    {
        if (minutes < 0) throw new ArgumentOutOfRangeException(nameof(minutes));
        WaitingMinutes = minutes;
        WasOnTime = minutes == 0;
    }
    public override void DisplayReview()
    {
        base.DisplayReview();
        Console.WriteLine($"Время: {StartTime:hh\\:mm}–{EndTime:hh\\:mm}; ожидание: {WaitingMinutes} мин.");
    }
    public override string GetReviewDetails() =>
        $"{base.GetReviewDetails()}; время: {StartTime:hh\\:mm}–{EndTime:hh\\:mm}; длительность: {CalculateDuration().TotalMinutes} мин.; ожидание: {WaitingMinutes} мин.; вовремя: {WasOnTime}; задержка начала: {CalculateDelay().TotalMinutes} мин.";
}

The below script needs to be able to find the current output cell; this is an easy method to get it.

In [2]:
public class ReviewCollection<T> where T : Review
{
    private readonly List<T> items = new List<T>();
    public int Count => items.Count;
    public void Add(T item)
    {
        if (item == null) throw new ArgumentNullException(nameof(item));
        items.Add(item);
    }
    public bool Remove(T item) => items.Remove(item);
    public void DisplayAll()
    {
        foreach (T item in items) { item.DisplayReview(); Console.WriteLine(item.GetReviewDetails()); }
    }
}

### Создание объектов и вызовы новых методов

In [3]:
var product = new ProductReview(1, "Удобная клавиатура", 4, "P-018", "Максим",
    "Клавиатура", new DateTime(2026, 9, 19));
product.RenameTitle("Покупка клавиатуры");
product.SetLanguage("ru");
product.EditText("Механическая клавиатура", "Удобные клавиши и приятная подсветка");
product.SetPros("Удобные клавиши");
product.SetCons("Короткий кабель");
product.Recommend(true);
product.ConfirmPurchase();
product.RateProduct(5);       // Унаследованный метод с одним параметром.
product.RateProduct(5, 4);    // Перегрузка с двумя параметрами: результат 5.

var service = new ServiceReview(2, "Хорошая стрижка", 4, new DateTime(2026, 9, 20),
    "Анна", "Стрижка", "Елена", 1000m);
service.SetLocation("Тюмень, ул. Республики, 10");
service.SetBookingCode("S-018");
service.RecommendService(true);

var timed = new ServiceTimeReview(3, "Быстро отремонтировали телефон", 5,
    new DateTime(2026, 9, 21), new TimeSpan(14, 0, 0), "Олег",
    "Ремонт телефона", "Иван", 1500m, new TimeSpan(15, 0, 0));
timed.SetLocation("Тюмень, ул. Ленина, 5");
timed.SetBookingCode("T-018");
timed.RecommendService(true);
timed.SetArrival(new TimeSpan(14, 0, 0), new TimeSpan(14, 10, 0));
timed.RecordWaiting(10);
Console.WriteLine($"Задержка: {timed.CalculateDelay().TotalMinutes} мин.");
timed.ResetArrival();
Console.WriteLine($"После сброса задержки: {timed.CalculateDelay().TotalMinutes} мин.");
timed.SetArrival(new TimeSpan(14, 0, 0), new TimeSpan(14, 10, 0));

var reviews = new ReviewCollection<Review>();
reviews.Add(product);
reviews.Add(service);
reviews.Add(timed);
foreach (Review review in new Review[] { product, service, timed })
{
    IPublishable publication = review;
    publication.Publish();
    review.MarkHelpful();
    Console.WriteLine(review.GetReviewDetails(true));  // Перегрузка: краткая форма.
    Console.WriteLine(review.GetReviewDetails(false)); // Через виртуальный метод наследника.
}
Console.WriteLine("Полиморфный вывод generic-коллекции:");
reviews.DisplayAll();

var products = new ReviewCollection<ProductReview>();
products.Add(product);
products.DisplayAll();
// products.Add(service); // Ошибка компиляции: требуется ProductReview.
Console.WriteLine($"Удаление услуги: {reviews.Remove(service)}; осталось: {reviews.Count}");
Console.WriteLine($"Повторное удаление: {reviews.Remove(service)}");

Задержка: 10 мин.


После сброса задержки: 0 мин.


Механическая клавиатура: 5/5


Механическая клавиатура [ru], обновлён: 02.11.2026 03:24; отзыв #1: Удобные клавиши и приятная подсветка; 5/5; автор: Максим; создан: 02.11.2026; опубликован: True; полезен: 1; товар: P-018, Клавиатура; покупка: 19.09.2026; подтверждена: True; плюсы: Удобные клавиши; минусы: Короткий кабель; рекомендую: True


Отзыв: 4/5


Отзыв [ru], обновлён: 02.11.2026 03:24; отзыв #2: Хорошая стрижка; 4/5; автор: Анна; создан: 02.11.2026; опубликован: True; полезен: 1; услуга: Стрижка; посещение: 20.09.2026; сотрудник: Елена; стоимость: 1000,00 руб.; адрес: Тюмень, ул. Республики, 10; запись: S-018; вернусь: True


... (10 outputs left out)

Товар «Клавиатура»: Удобные клавиши и приятная подсветка (5/5)


Механическая клавиатура [ru], обновлён: 02.11.2026 03:24; отзыв #1: Удобные клавиши и приятная подсветка; 5/5; автор: Максим; создан: 02.11.2026; опубликован: True; полезен: 1; товар: P-018, Клавиатура; покупка: 19.09.2026; подтверждена: True; плюсы: Удобные клавиши; минусы: Короткий кабель; рекомендую: True


Удаление услуги: True; осталось: 2


Повторное удаление: False
